# 07 Metrics

Pooled and per-station metrics for every method, station-level bootstrap intervals, the M9 confidence-versus-coverage table, the release-gate decision, the Beta `unsure` sensitivity table and the M9 calibration reliability.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** `06_site_days/site_days.parquet`.

**Outputs.** `outputs/01_final_evaluation/07_metrics/`: `pooled.csv`, `stations.csv`, `bootstrap.csv`, `coverage.csv`, `gate.json`, `sensitivity.csv`, `calibration_reliability.csv`; `manifests/07_metrics.json`.

**Approximate runtime.** About one minute (the bootstrap is 1,000 station draws per method and group).

**Prerequisites.** Notebook 06.

**Main process.**

1. Pool energies by summing before dividing; the four headline metrics are Reference Energy IoU, reference energy precision, site-day F1 and site-day precision; supporting metrics follow section 04 of the evaluation methodology.
2. Bootstrap stations with replacement (seed 9) for 2.5–97.5 percentile intervals.
3. Re-decide M9 at every c in the grid for the coverage table.
4. Apply the gate: reference energy precision ≥ 0.90 on Beta `sure`; Alpha reported against its label-contiguity ceiling.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Compute

In [ ]:
result = cli.stage_metrics(SETTINGS)
pooled = result["pooled"]
display(pooled[["method", "group", "n_stations", "n_days", "n_rpf", "energy_iou", "energy_precision", "day_f1", "day_precision",
                "day_recall", "sure_day_recall", "rate_uncertain"]].round(4))

## 3. Uncertainty and gate

The intervals resample stations, the unit of generalisation, so they are wide where a few stations carry most of the reference energy. The gate decision is recorded in `gate.json`.

In [ ]:
display(result["bootstrap"].round(4))
display(pd.Series(result["gate"]["methods"]).apply(pd.Series).round(4))
print(result["gate"]["decision"])

## 4. Confidence versus coverage (M9)

For each value of the public control c: the share of days decided automatically, the days sent to review, and the errors among automatic decisions.

In [ ]:
display(result["coverage"].round(4))

## 5. Sensitivity: Beta `unsure`

Reported only; never used for selection, tuning or the gate.

In [ ]:
display(result["sensitivity"][["method", "group", "n_days", "n_rpf", "energy_iou", "energy_precision", "day_f1", "day_precision"]].round(4))

## Conclusion

The metric tables are frozen files; notebook 08 renders them and never recomputes them.